In [1]:
import requests
import pandas as pd
import time
from datetime import datetime

def get_openaire_articles(keywords, page_size=25, max_pages=2, open_access_only=True):
    """
    Pobiera artykuły z OpenAIRE Search API (używając 'keywords' zamiast 'query').
    """
    print(f"Rozpoczynam pobieranie dla słów kluczowych (Search API): '{keywords}'...")
    
    # Poprawny endpoint
    base_url = "https://api.openaire.eu/search/publications"
    
    all_records = []
    current_page = 1
    
    for page in range(1, max_pages + 1):
        params = {
            "keywords": keywords,  ### Zmienione z 'query'
            "size": page_size,
            "page": current_page,  ### Paginacja przez page (1-based)
        }
        if open_access_only:
            params["openAccessColor"] = "gold"  # tylko otwarte publikacje, pełne Open Access (bezpieczne prawnie)
            params["format"] = "json" # odpowiedź w JSON (łatwiejsze parsowanie)
        
        print(f"  > Wysyłane parametry: {params}")
        
        try:
            response = requests.get(base_url, params=params, headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'})
            response.raise_for_status()
            
            data = response.json()
            # print(f"  > DEBUG JSON: {str(data)[:200]}...")

            response_part = data.get('response', {})
            
            results = response_part.get('results', {})

            ## zabezpieczenie: Czasem API zwraca None zamiast {} gdy nie ma wyników na danej stronie
            if results is None:
                print(f"  > Brak dalszych wyników na stronie {page} (koniec).")
                break

            raw_records = results.get('result', [])
            

            ### OpenAIRE API jest dziwne: jak jest 1 wynik to zwraca dict, jak więcej to listę ### trzeba to ujednolicić do listy:
            if isinstance(raw_records, dict):
                records = [raw_records]
            elif isinstance(raw_records, list):
                records = raw_records
            else:
                records = []
                
            
            if not records:
                print(f"  > Brak rekordów (struktura pusta).")
                break
                
            print(f"  > Znaleziono {len(records)} rekordów na stronie {page}.")
                
            for record in records:
                try:
                    oaf_result = record.get('metadata', {}).get('oaf:entity', {}).get('oaf:result', {})

                    # Tytuł
                    title = ''
                    title_list = oaf_result.get('title', [])
                    if title_list and isinstance(title_list, list):
                        first = title_list[0]
                        title = first if isinstance(first, str) else (first.get('$', '') if isinstance(first, dict) else '')
                    elif isinstance(title_list, dict): # czasem tytuł to dict
                        title = title_list.get('$', '')
                    
                    # Abstrakt (description -> często string lub dict)
                    abstract = ''
                    desc = oaf_result.get('description', '')
                    if isinstance(desc, str):
                        abstract = desc
                    elif isinstance(desc, dict):
                        abstract = desc.get('$', '')

                    # Fallback na abstract (rzadko)
                    if not abstract:
                        abs_list = oaf_result.get('abstract', [])
                        if abs_list and isinstance(abs_list, list):
                            first = abs_list[0]
                            abstract = first if isinstance(first, str) else (first.get('$', '') if isinstance(first, dict) else '')
                    
                    # Rok
                    pub_date = oaf_result.get('dateofacceptance', '') or (oaf_result.get('relevantdate', [{}])[0].get('$', '') if oaf_result.get('relevantdate') else '')
                    year = pub_date[:4] if pub_date and isinstance(pub_date, str) else ''
                    if isinstance(pub_date, dict): year = pub_date.get('$', '')[:4]
                    
                    # Licencja
                    access = oaf_result.get('bestaccessright', {})
                    license_name = access.get('@classid', '') if isinstance(access, dict) else ''
                    
                    # Link
                    link = ''
                    instances = oaf_result.get('children', {}).get('instance', [])
                    if instances and isinstance(instances, list) and len(instances) > 0:
                        webresources = instances[0].get('webresource', [])
                        # czasem webresource to lista, czasem dict
                        if isinstance(webresources, list) and len(webresources) > 0:
                            url_obj = webresources[0].get('url', {})
                            link = url_obj if isinstance(url_obj, str) else (url_obj.get('$', '') if isinstance(url_obj, dict) else '')
                        elif isinstance(webresources, dict):
                             url_obj = webresources.get('url', {})
                             link = url_obj if isinstance(url_obj, str) else (url_obj.get('$', '') if isinstance(url_obj, dict) else '')

                    # Fallback: DOI
                    if not link:
                        pid_list = oaf_result.get('pid', [])
                        # pid_list może być dictem jeśli jest tylko jeden PID
                        if isinstance(pid_list, dict): pid_list = [pid_list]
                        
                        if isinstance(pid_list, list):
                            for pid in pid_list:
                                if isinstance(pid, dict) and pid.get('@classid') == 'doi':
                                    doi = pid.get('$', '')
                                    if doi:
                                        link = f"https://doi.org/{doi}"
                                        break
                    
                    # Journal title
                    source = oaf_result.get('source', '')
                    journal_title = ''
                    if isinstance(source, str):
                        journal_title = source
                    elif isinstance(source, list) and len(source) > 0:
                        journal_title = source[0] if isinstance(source[0], str) else (source[0].get('$', '') if isinstance(source[0], dict) else '')

                    # Authors
                    authors_str = ""
                    creators = oaf_result.get('creator', [])
                    
                    authors_list = []
                    ## przypadek 1: lista autorów
                    if isinstance(creators, list):
                        for c in creators:
                            if isinstance(c, dict):
                                ### imię i nazwisko jest zazwyczaj pod kluczem '$'
                                name = c.get('$', '')
                                if name:
                                    authors_list.append(name)
                            elif isinstance(c, str):
                                authors_list.append(c)
                    
                    ## przypadek 2: pojedynczy autor (dict)
                    elif isinstance(creators, dict):
                        name = creators.get('$', '')
                        if name:
                            authors_list.append(name)
                    
                    ## przypadek 3: pojedynczy autor (str)
                    elif isinstance(creators, str):
                        authors_list.append(creators)

                    ## łączenie średnikami
                    authors_str = "; ".join(authors_list)
                    
                    # dodaje tylko jesli jest abstract
                    if abstract and abstract.strip():
                        all_records.append({
                            "keywords": keywords,
                            "title": title.strip() if title else "",
                            "creator": authors_str,
                            "abstract": abstract.strip(),
                            "year": year,
                            "journal_title": journal_title,
                            "license": license_name,
                            "link": link or "",
                            "query": keywords
                        })

                except Exception as e:
                    print(f"  > Błąd parsowania rekordu: {e}")
                    continue
            
            current_page += 1
            
        except requests.exceptions.RequestException as e:
            print(f"Błąd sieci: {e}")
            break
        
        # przerwa między stronami
        time.sleep(2)
    
    print(f"Pobrano {len(all_records)} artykułów z abstraktami dla słów kluczowych: '{keywords}'.")
    return all_records

# Główna część skryptu
if __name__ == "__main__":
    
    KEYWORDS_TO_SEARCH = [
        "ayyubid",
        "fall of rome",
        "sack of rome 410",
        "sack of rome 455",
        "battle of chalons",
        "theoderic",
        "justinian reconquest",
        "plague of justinian",
        "maurice strategikon",
        "byzantine empire 5th century",
        "western roman empire collapse",
        "romanitas",
        "transformation of the roman world",
        "pirenne thesis",
        "romano-germanic kingdoms",
        "post-roman",
        "sub-roman britain",
        "anglo-saxon settlement",
        "frisian migration",
        "slavic expansion",
        "avar khaganate",
        "lombard italy",
        "cassiodorus",
        "jordanes",
        "gregory of tours",
        "isidore of seville",
        "roman law codification",
        "theodosian code",
        "salic law",
        "gothic spain",
        "alan kingdom",
        "vandal africa",
        "roman identity 5th-7th",
        "early medieval history",
        "barbarian migrations", 
        "vikings", 
        "carolingian",
        "merovingian",
        "early middle ages",
        "byzantium",
        "dark ages",
        "anglo-saxon history",
        "saxons",
        "angles in britannia",
        "jutes",
        "early slavs",
        "romano-britons",
        "romano-british",
		"huns history",
		"history huns",
		"white huns",
		"black huns",
        "goths",
        "ostrogoths",
        "visigoths",
        "history franks",
		"franks history",
        "alemanni history",
		"history alemanni",
        "vandals history",
        "gepids",
        "lombards",
        "attila",
        "thuringians",
        "rugians",
        "sciri",
        "herules",
        "bavarians history",
        "avars history",
        "burgundians",
        "viking",
        "germanic paganism",
        "slavic paganism",
        "medieval celts",
        "norse paganism",
        "foederati",
        "roman britannia",
        "suebi history",
        "alans history",
		"history alans",
        "frisians",
        "sarmatians",
        "period 400-1000",
        "late roman empire",
        "barbarian kingdoms",
        "arianism christianity",
        "magyars",
        "khazar",
		"khazars",
        "umayyad",
        "charlemagne",
        "clovis",
        "theodoric the great",
        "medieval constantinople",
        "gothic war",
        "barbarian invasions",
        "rome imperial borders",
        "late antiquity",
        "alaric",
        "fall of the western roman empire",
        "imperial restoration",
        "justinian",
        "odoacer",
        "christianization",
        "migration period",
        "eastern roman empire",
        "medieval rome",
        "early bulgars",
        "danelag",
		"danelaw",
        "visigothic spain",
        "king arthur history",
        "norman kingdom",
		"hephthalite",
		"ephthalite",
		"ebodalo",
        "nomadic empire",
        "kidarites",
        "alchon huns",
        "kushan",
        "rouran",
        "xianbei",
        "xiongnu",
        "yuezhi",
        "saka people",
        "chionites",
        "bactria",
        "roman gaul",
        "sasanian",
        "persian empire",
        "latin empire",
        "early danes",
        "beowulf",
        "roman civil wars",
        "crossing of the rhine",
        "rome imperial government",
        "barbarian rulers",
        "roman heritage",
        "roman emperors",
        "nicene christianity",
        "gothic kingdoms",
        "han dynasty",
        "tang china",
        "roman usurpers",
        "magister militum",
        "late roman army",
        "picts",
        "dal riada",
		"dal riata",
		"dalriada",
        "gaelic tribes",
        "heptarchy",
        "bretwalda",
        "old english period",
        "kingdom of east anglia",
        "germanic settlement",
        "geats",
		"vistula venedi",
        "venedi slavs",
        "wends history",
        "sclaveni",
        "antes slavs",
        "norsemen",
        "balto-slavic",
        "proto-slavic",
        "slavic settlement",
        "getica",
        "jordanes",
        "sarmatae",
        "widsith",
        "king samo",
		"samo empire",
        "volga bulgaria",
        "varangian",
        "old great bulgaria",
        "crusades",
        "saqaliba",
        "golden horde",
        "kara-khanid",
        "seljuk",
        "kara khitai",
        "great liao",
        "khitan",
        "kipchak",
        "qangli",
        "kangly",
        "cumans history",
        "pechenegs",
        "karluks",
        "gokturks",
        "hazaras",
        "chagatai",
        "turco-mongol",
        "ilkhanate",
        "timurids",
        "jalayirid",
        "sultanate of rum",
        "turco-persian",
        "anatolian beyliks",
        "cilician armenia",
        "empire of trebizond",
        "yuan china",
        "hunnic empire",
        "vistula veneti",
        "karamanids",
        "aq qoyunlu",
        "mamluks",
        "qara qoyunlu",
        "teke beylik",
        "aydin beylik",
        "menteshe",
        "danishmend",
        "mengujekids",
        "saltukids",
        "kayi tribe",
        "ghaznavids",
        "great moravia",
        "kievan rus",
        "baltic slavic piracy",
        "rashidun caliphate",
        "abbasid",
        "muslim conquest of persia",
        "muslim conquest of the levant",
        "emperor heraclius",
        "muslim conquest of egypt",
        "muslim conquest of maghreb",
        "ghassanids",
        "lakhmids",
        "assyrian church",
        "nestorians",
        "paulicianism",
        "bogomilism",
        "tondrakians",
        "himyar",
        "lotharingia",
        "neustria",
        "austrasia",
        "kingdom of soissons",
        "ambrosius aurelianus",
        "gildas",
        "arthurian period",
        "de excidio et conquestu britanniae",
        "medieval brittany",
        "kingdom of galicia",
        "marcomanni",
        "marcus aurelius",
        "ricimer",
        "aetius",
        "majorian",
        "aegidius",
        "syagrius",
        "crisis of third century",
        "constantine the great",
        "catalaunian plains",
        "ripuarian franks",
        "salian franks",
        "ammianus marcellinus",
        "burgundy history",
		"history burgundy",
        "strategikon",
        "roxolani",
        "iazyges",
        "sabirs",
        "onoghurs",
        "utigurs",
        "kutrigurs",
        "akatziri",
        "barsils",
        "mishar tatars",
        "mordvins",
        "qaraqalpaqs",
        "nogai horde",
        "bashkirs",
        "tengrism",
        "uyghur khaganate",
        "yenisei kyrgyz",
    ]
    
    ARTICLES_PER_KEYWORD = 50
    MAX_PAGES_PER_KEYWORD = 3  # do 150 wyników na keyword
    
    all_articles_data = []
    
    print("Rozpoczynam zbieranie danych z OpenAIRE (Search API)...")
    
    for keyword in KEYWORDS_TO_SEARCH:
        articles = get_openaire_articles(keyword, page_size=ARTICLES_PER_KEYWORD, max_pages=MAX_PAGES_PER_KEYWORD)
        all_articles_data.extend(articles)
        
        # przerwa między keywordami
        time.sleep(3) 
    
    if all_articles_data:
        df = pd.DataFrame(all_articles_data)
        
        ## output_filename = f"openaire_articles_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        output_filename = "openaire_articles_updated.csv"
        df.to_csv(output_filename, index=False, encoding='utf-8')
        
        print(f"\nGotowe! Zapisano łącznie {len(df)} artykułów do pliku: {output_filename}")
        print("Przykładowe dane:")
        print(df.head())
    else:
        print("Nie udało się pobrać żadnych artykułów.")

Rozpoczynam zbieranie danych z OpenAIRE (Search API)...
Rozpoczynam pobieranie dla słów kluczowych (Search API): 'ayyubid'...
  > Wysyłane parametry: {'keywords': 'ayyubid', 'size': 50, 'page': 1, 'openAccessColor': 'gold', 'format': 'json'}
  > Znaleziono 50 rekordów na stronie 1.
  > Wysyłane parametry: {'keywords': 'ayyubid', 'size': 50, 'page': 2, 'openAccessColor': 'gold', 'format': 'json'}
  > Znaleziono 50 rekordów na stronie 2.
  > Wysyłane parametry: {'keywords': 'ayyubid', 'size': 50, 'page': 3, 'openAccessColor': 'gold', 'format': 'json'}
  > Znaleziono 9 rekordów na stronie 3.
Pobrano 80 artykułów z abstraktami dla słów kluczowych: 'ayyubid'.
Rozpoczynam pobieranie dla słów kluczowych (Search API): 'fall of rome'...
  > Wysyłane parametry: {'keywords': 'fall of rome', 'size': 50, 'page': 1, 'openAccessColor': 'gold', 'format': 'json'}
  > Znaleziono 50 rekordów na stronie 1.
  > Wysyłane parametry: {'keywords': 'fall of rome', 'size': 50, 'page': 2, 'openAccessColor': 'gold

# quickfix

In [ ]:
import pandas as pd
import re

# 1. Wczytanie obecnego pliku
input_file = "openaire_articles_updated.csv"
output_file = "openaire_articles_updated_fixed.csv"

print(f"Wczytuję {input_file}...")
try:
    df = pd.read_csv(input_file)
    initial_count = len(df)
    print(f"Liczba wierszy na start: {initial_count}")
except FileNotFoundError:
    print("Nie znaleziono pliku wejściowego!")
    df = pd.DataFrame()

if not df.empty:
    # 2. Usuwanie tagów HTML/XML (Quickfix)
    ### wyrażenie r'<[^>]+>' oznacza: znajdź "<", potem cokolwiek co NIE jest ">", aż do zamknięcia ">"
    def clean_tags(text):
        if isinstance(text, str):
            return re.sub(r'<[^>]+>', '', text)
        return text

    print("Usuwam tagi HTML/XML z tytułów i abstraktów...")
    cols_to_clean = ['title', 'abstract']
    for col in cols_to_clean:
        if col in df.columns:
            df[col] = df[col].apply(clean_tags)

    # 3. Deduplikacja
    ### usuwanie duplikatów sprawdzając tylko kombinację tytuł+autor+abstrakt
    ### keep='first' -> zostawia pierwsze wystąpienie (z pierwszym przypisanym keywordem), resztę usuwa
    print("Rozpoczynam deduplikację...")
    df = df.drop_duplicates(subset=['title', 'creator', 'abstract'], keep='first')
    
    final_count = len(df)
    print(f"Usunięto {initial_count - final_count} duplikatów.")
    print(f"Liczba unikalnych artykułów: {final_count}")

    # 4. Zapis
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"\nZapisano czysty plik: {output_file}")
    
    ## podgląd po czyszczeniu
    print("\nPrzykładowy wyczyszczony abstrakt:")
    print(df['abstract'].iloc[0][:200] + "...")